In [16]:
pip install pandas numpy scikit-learn shap lime matplotlib xgboost imbalanced-learn

In [1]:
import os
import time
import timeit
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, f1_score, precision_recall_curve,
    average_precision_score, roc_curve
)
from imblearn.over_sampling import SMOTE
import shap
from lime import lime_tabular

warnings.filterwarnings('ignore')

/Users/noobkirati/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Load the dataset
data = pd.read_csv('creditcard.csv')

data = data.dropna(subset=['Class'])
X = data.drop('Class', axis=1)
y = data['Class']


print("=== Dataset Information ===")
print(f"Shape: {data.shape}  |  Fraud cases: {y.sum()}  |  Imbalance: {y.mean()*100:.3f}%")


=== Dataset Information ===
Shape: (81298, 31)  |  Fraud cases: 198.0  |  Imbalance: 0.244%


In [33]:
print("NaNs in y:", y.isna().sum())

NaNs in y: 0


In [ ]:
# Preprocess the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

print(f"Train after SMOTE: {len(y_train_res)} | Class balance: "
      f"{pd.Series(y_train_res).value_counts(normalize=True).to_dict()}")

feature_names = X.columns.tolist()


Train after SMOTE: 129760 | Class balance: {0.0: 0.5, 1.0: 0.5}


In [ ]:
# Helper functions for evaluation and plotting
def save_fig(filename):
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {filename}")

def plot_confusion_matrix(cm, title, filename):
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Legitimate', 'Fraud'],
                yticklabels=['Legitimate', 'Fraud'])
    plt.title(title)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    save_fig(filename)

def evaluate_model(name, y_true, y_pred, y_proba):
    """Return a dict of all key metrics."""
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    return {
        'Model':     name,
        'ROC-AUC':   round(roc_auc_score(y_true, y_proba), 4),
        'PR-AUC':    round(average_precision_score(y_true, y_proba), 4),
        'F1-Fraud':  round(f1_score(y_true, y_pred), 4),
        'Precision': round(tp / (tp + fp) if (tp + fp) > 0 else 0, 4),
        'Recall':    round(tp / (tp + fn) if (tp + fn) > 0 else 0, 4),
        'FP':        int(fp),
        'FN':        int(fn),
        'TP':        int(tp),
        'TN':        int(tn),
    }

In [ ]:
# Model 1: logistic regression (baseline)
print("\n" + "="*65)
print("MODEL 1: LOGISTIC REGRESSION (Baseline)")
print("="*65)

model1 = LogisticRegression(max_iter=1000, random_state=42)
model1.fit(X_train_res, y_train_res)

y_pred1       = model1.predict(X_test_scaled)
y_proba1      = model1.predict_proba(X_test_scaled)[:, 1]

print(classification_report(y_test, y_pred1, target_names=['Legitimate', 'Fraud']))
cm1 = confusion_matrix(y_test, y_pred1)
plot_confusion_matrix(cm1, 'Confusion Matrix — Model 1 (Logistic Regression)', 'ConfusionMAT1.png')
metrics1 = evaluate_model('Logistic Regression', y_test, y_pred1, y_proba1)
print(f"  ROC-AUC: {metrics1['ROC-AUC']}  PR-AUC: {metrics1['PR-AUC']}  F1: {metrics1['F1-Fraud']}")



MODEL 1: LOGISTIC REGRESSION (Baseline)
              precision    recall  f1-score   support

  Legitimate       1.00      0.98      0.99     16220
       Fraud       0.09      0.95      0.17        40

    accuracy                           0.98     16260
   macro avg       0.55      0.96      0.58     16260
weighted avg       1.00      0.98      0.99     16260

  Saved: ConfusionMAT1.png
  ROC-AUC: 0.9893  PR-AUC: 0.5848  F1: 0.167


In [ ]:
# Model 2: XGBoost — RandomizedSearchCV (tuned)
print("\n" + "="*65)
print("MODEL 2: XGBoost (Tuned)")
print("="*65)

from xgboost import XGBClassifier

xgb_param_dist = {
    'n_estimators':  [100, 200, 300],
    'max_depth':     [3, 5, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample':     [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
    'min_child_weight': [1, 3, 5],
}

xgb_base = XGBClassifier(
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

xgb_search = RandomizedSearchCV(
    xgb_base, xgb_param_dist,
    n_iter=20,
    scoring='f1',
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
    random_state=42,
    n_jobs=-1,
    verbose=0
)
print("  Running RandomizedSearchCV for XGBoost (20 iterations, 3-fold CV)...")
xgb_search.fit(X_train_res, y_train_res)
print(f"  Best params: {xgb_search.best_params_}")
print(f"  Best CV F1:  {xgb_search.best_score_:.4f}")

model2 = xgb_search.best_estimator_
y_pred2  = model2.predict(X_test_scaled)
y_proba2 = model2.predict_proba(X_test_scaled)[:, 1]

print(classification_report(y_test, y_pred2, target_names=['Legitimate', 'Fraud']))
cm2 = confusion_matrix(y_test, y_pred2)
plot_confusion_matrix(cm2, 'Confusion Matrix — Model 2 (XGBoost)', 'ConfusionMAT2.png')
metrics2 = evaluate_model('XGBoost (Tuned)', y_test, y_pred2, y_proba2)
print(f"  ROC-AUC: {metrics2['ROC-AUC']}  PR-AUC: {metrics2['PR-AUC']}  F1: {metrics2['F1-Fraud']}")



MODEL 2: XGBoost (Tuned)
  Running RandomizedSearchCV for XGBoost (20 iterations, 3-fold CV)...
  Best params: {'subsample': 0.8, 'n_estimators': 300, 'min_child_weight': 3, 'max_depth': 8, 'learning_rate': 0.1, 'colsample_bytree': 0.8}
  Best CV F1:  0.9998
              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00     16220
       Fraud       0.90      0.95      0.93        40

    accuracy                           1.00     16260
   macro avg       0.95      0.97      0.96     16260
weighted avg       1.00      1.00      1.00     16260

  Saved: ConfusionMAT2.png
  ROC-AUC: 0.9919  PR-AUC: 0.9613  F1: 0.9268


In [ ]:
# MODEL 3: RANDOM FOREST — RandomizedSearchCV tuning
print("\n" + "="*65)
print("MODEL 3: RANDOM FOREST (Tuned)")
print("="*65)

rf_param_dist = {
    'n_estimators':      [100, 200, 300],
    'max_depth':         [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
    'max_features':      ['sqrt', 'log2'],
}

rf_base = RandomForestClassifier(random_state=42, n_jobs=-1)

rf_search = RandomizedSearchCV(
    rf_base, rf_param_dist,
    n_iter=20,
    scoring='f1',
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
    random_state=42,
    n_jobs=-1,
    verbose=0
)
print("  Running RandomizedSearchCV for Random Forest (20 iterations, 3-fold CV)...")
rf_search.fit(X_train_res, y_train_res)
print(f"  Best params: {rf_search.best_params_}")
print(f"  Best CV F1:  {rf_search.best_score_:.4f}")

model3 = rf_search.best_estimator_
y_pred3  = model3.predict(X_test_scaled)
y_proba3 = model3.predict_proba(X_test_scaled)[:, 1]

print(classification_report(y_test, y_pred3, target_names=['Legitimate', 'Fraud']))
cm3 = confusion_matrix(y_test, y_pred3)
plot_confusion_matrix(cm3, 'Confusion Matrix — Model 3 (Random Forest)', 'ConfusionMAT3.png')
metrics3 = evaluate_model('Random Forest (Tuned)', y_test, y_pred3, y_proba3)
print(f"  ROC-AUC: {metrics3['ROC-AUC']}  PR-AUC: {metrics3['PR-AUC']}  F1: {metrics3['F1-Fraud']}")



MODEL 3: RANDOM FOREST (Tuned)
  Running RandomizedSearchCV for Random Forest (20 iterations, 3-fold CV)...
  Best params: {'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 30}
  Best CV F1:  0.9999
              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00     16220
       Fraud       0.95      0.93      0.94        40

    accuracy                           1.00     16260
   macro avg       0.97      0.96      0.97     16260
weighted avg       1.00      1.00      1.00     16260

  Saved: ConfusionMAT3.png
  ROC-AUC: 0.9847  PR-AUC: 0.9628  F1: 0.9367


In [ ]:
# Model comparison table (default threshold)
print("\n" + "="*65)
print("MODEL COMPARISON — Default threshold (0.5)")
print("="*65)
comparison_df = pd.DataFrame([metrics1, metrics2, metrics3])
print(comparison_df[['Model','ROC-AUC','PR-AUC','F1-Fraud',
                      'Precision','Recall','FP','FN']].to_string(index=False))
comparison_df.to_csv('model_comparison_default.csv', index=False)
print("  Saved: model_comparison_default.csv")



MODEL COMPARISON — Default threshold (0.5)
                Model  ROC-AUC  PR-AUC  F1-Fraud  Precision  Recall  FP  FN
  Logistic Regression   0.9893  0.5848    0.1670     0.0916   0.950 377   2
      XGBoost (Tuned)   0.9919  0.9613    0.9268     0.9048   0.950   4   2
Random Forest (Tuned)   0.9847  0.9628    0.9367     0.9487   0.925   2   3
  Saved: model_comparison_default.csv


In [ ]:
# Precision-Recall curves (all 3 models on one plot)
print("\n  Plotting PR curves...")
plt.figure(figsize=(8, 6))
for proba, name, ls in [
    (y_proba1, f"Logistic Regression (PR-AUC={metrics1['PR-AUC']})", '--'),
    (y_proba2, f"XGBoost           (PR-AUC={metrics2['PR-AUC']})", '-.'),
    (y_proba3, f"Random Forest     (PR-AUC={metrics3['PR-AUC']})", '-'),
]:
    p, r, _ = precision_recall_curve(y_test, proba)
    plt.plot(r, p, linestyle=ls, label=name, linewidth=1.8)



  Plotting PR curves...


In [41]:
# Baseline (random classifier)
plt.axhline(y=y_test.mean(), color='gray', linestyle=':', linewidth=1, label='Baseline (random)')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision–Recall Curves — All Models')
plt.legend(fontsize=9)
plt.grid(alpha=0.3)
save_fig('pr_curves.png')

  Saved: pr_curves.png


In [42]:
# ROC curves
plt.figure(figsize=(8, 6))
for proba, name, ls in [
    (y_proba1, f"Logistic Regression (AUC={metrics1['ROC-AUC']})", '--'),
    (y_proba2, f"XGBoost           (AUC={metrics2['ROC-AUC']})", '-.'),
    (y_proba3, f"Random Forest     (AUC={metrics3['ROC-AUC']})", '-'),
]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    plt.plot(fpr, tpr, linestyle=ls, label=name, linewidth=1.8)
plt.plot([0,1],[0,1], color='gray', linestyle=':', linewidth=1, label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — All Models')
plt.legend(fontsize=9)
plt.grid(alpha=0.3)
save_fig('roc_curves.png')

  Saved: roc_curves.png


In [ ]:
# THRESHOLD OPTIMISATION & FALSE POSITIVE REDUCTION  (RQ3)
# Apply to all models, uses Random Forest as the primary demonstration.
print("\n" + "="*65)
print("THRESHOLD OPTIMISATION & FP REDUCTION ANALYSIS (RQ3)")
print("="*65)

def optimise_threshold(y_true, y_proba, model_name):
    """
    Find the threshold that maximises F1 and compare FPs at:
      - default threshold (0.5)
      - F1-maximising threshold
      - 80% recall fixed threshold
    Returns a summary dict and prints results.
    """
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_proba)

    # F1-maximising threshold
    f1_scores = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-9)
    best_idx  = np.argmax(f1_scores)
    best_thresh = thresholds[best_idx]

    # 80%-recall threshold
    recall_idx  = np.argmax(recalls[:-1] >= 0.80)
    recall_thresh = thresholds[recall_idx]

    results = {}
    print(f"\n  {model_name}")
    print(f"  {'Threshold':<18} {'Threshold Val':<16} {'TP':<6} {'FP':<6} {'FN':<6} {'F1':<8} {'Recall':<8}")
    print("  " + "-"*70)

    for label, thresh in [
        ('Default (0.5)',     0.5),
        ('F1-maximising',     best_thresh),
        ('Fixed recall 80%',  recall_thresh),
    ]:
        y_pred_t = (y_proba >= thresh).astype(int)
        cm_t = confusion_matrix(y_true, y_pred_t)
        tn_t, fp_t, fn_t, tp_t = cm_t.ravel()
        f1_t  = f1_score(y_true, y_pred_t)
        rec_t = tp_t / (tp_t + fn_t) if (tp_t + fn_t) > 0 else 0
        print(f"  {label:<18} {thresh:<16.4f} {tp_t:<6} {fp_t:<6} {fn_t:<6} {f1_t:<8.4f} {rec_t:<8.4f}")
        results[label] = {'threshold': thresh, 'TP': tp_t, 'FP': fp_t,
                          'FN': fn_t, 'F1': f1_t, 'Recall': rec_t}

    # FP reduction from default to F1-optimal
    fp_default = results['Default (0.5)']['FP']
    fp_optimal = results['F1-maximising']['FP']
    fp_reduction_pct = (fp_default - fp_optimal) / fp_default * 100 if fp_default > 0 else 0
    print(f"\n  FP reduction (default → F1-optimal): "
          f"{fp_default} → {fp_optimal} ({fp_reduction_pct:.1f}% reduction)")
    results['fp_reduction_pct'] = fp_reduction_pct
    results['best_thresh'] = best_thresh
    return results

thresh_results1 = optimise_threshold(y_test, y_proba1, 'Logistic Regression')
thresh_results2 = optimise_threshold(y_test, y_proba2, 'XGBoost')
thresh_results3 = optimise_threshold(y_test, y_proba3, 'Random Forest')


THRESHOLD OPTIMISATION & FP REDUCTION ANALYSIS (RQ3)

  Logistic Regression
  Threshold          Threshold Val    TP     FP     FN     F1       Recall  
  ----------------------------------------------------------------------
  Default (0.5)      0.5000           38     377    2      0.1670   0.9500  
  F1-maximising      0.9998           32     11     8      0.7711   0.8000  
  Fixed recall 80%   0.0000           40     16220  0      0.0049   1.0000  

  FP reduction (default → F1-optimal): 377 → 11 (97.1% reduction)

  XGBoost
  Threshold          Threshold Val    TP     FP     FN     F1       Recall  
  ----------------------------------------------------------------------
  Default (0.5)      0.5000           38     4      2      0.9268   0.9500  
  F1-maximising      0.8399           37     1      3      0.9487   0.9250  
  Fixed recall 80%   0.0000           40     16220  0      0.0049   1.0000  

  FP reduction (default → F1-optimal): 4 → 1 (75.0% reduction)

  Random Forest
  

In [ ]:
# Threshold-F1 curve for Random Forest
precisions3, recalls3, thresholds3 = precision_recall_curve(y_test, y_proba3)
f1_over_thresh = 2 * precisions3[:-1] * recalls3[:-1] / (precisions3[:-1] + recalls3[:-1] + 1e-9)

plt.figure(figsize=(9, 4))
plt.subplot(1, 2, 1)
plt.plot(thresholds3, f1_over_thresh, color='steelblue', linewidth=1.8)
plt.axvline(thresh_results3['best_thresh'], color='crimson', linestyle='--',
            label=f"Best threshold ({thresh_results3['best_thresh']:.3f})")
plt.xlabel('Decision threshold')
plt.ylabel('F1-score (fraud class)')
plt.title('F1 vs Threshold — Random Forest')
plt.legend(fontsize=9)
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(thresholds3, precisions3[:-1], label='Precision', linewidth=1.8)
plt.plot(thresholds3, recalls3[:-1],    label='Recall',    linewidth=1.8)
plt.axvline(thresh_results3['best_thresh'], color='crimson', linestyle='--',
            label=f"Best threshold ({thresh_results3['best_thresh']:.3f})")
plt.xlabel('Decision threshold')
plt.title('Precision & Recall vs Threshold — Random Forest')
plt.legend(fontsize=9)
plt.grid(alpha=0.3)
save_fig('threshold_analysis_rf.png')

  Saved: threshold_analysis_rf.png


In [45]:
# Updated confusion matrix at optimal threshold for RF
best_thresh_rf = thresh_results3['best_thresh']
y_pred3_opt    = (y_proba3 >= best_thresh_rf).astype(int)
cm3_opt        = confusion_matrix(y_test, y_pred3_opt)
plot_confusion_matrix(cm3_opt,
    f'Confusion Matrix — RF (Optimal Threshold={best_thresh_rf:.3f})',
    'ConfusionMAT3_optimal.png')

  Saved: ConfusionMAT3_optimal.png


In [46]:
# Comparison table: default vs optimal threshold across models
print("\n  Threshold-optimised comparison table:")
opt_rows = []
for name, results, y_proba in [
    ('Logistic Regression', thresh_results1, y_proba1),
    ('XGBoost',             thresh_results2, y_proba2),
    ('Random Forest',       thresh_results3, y_proba3),
]:
    bt = results['best_thresh']
    y_p_opt = (y_proba >= bt).astype(int)
    m_opt   = evaluate_model(f"{name} (optimal)", y_test, y_p_opt, y_proba)
    opt_rows.append(m_opt)
opt_df = pd.DataFrame(opt_rows)
print(opt_df[['Model','ROC-AUC','PR-AUC','F1-Fraud','Precision','Recall','FP','FN']].to_string(index=False))
opt_df.to_csv('model_comparison_optimal.csv', index=False)
print("  Saved: model_comparison_optimal.csv")



  Threshold-optimised comparison table:
                        Model  ROC-AUC  PR-AUC  F1-Fraud  Precision  Recall  FP  FN
Logistic Regression (optimal)   0.9893  0.5848    0.7711     0.7442   0.800  11   8
            XGBoost (optimal)   0.9919  0.9613    0.9487     0.9737   0.925   1   3
      Random Forest (optimal)   0.9847  0.9628    0.9367     0.9487   0.925   2   3
  Saved: model_comparison_optimal.csv


In [ ]:
# SHAP EXPLAINERS — setup (computed once, reused for stability)
print("\n" + "="*65)
print("SHAP ANALYSIS")
print("="*65)

# SHAP explainers
explainer1 = shap.LinearExplainer(model1, X_train_res,
                                  feature_perturbation="interventional")
explainer2 = shap.TreeExplainer(model2)
explainer3 = shap.TreeExplainer(model3)

# Sample for global plots
X_sample_scaled = X_test_scaled[:500]
X_sample_df     = X_test.iloc[:500]

# Model 1: LR (old API is fine)
shap_values1 = explainer1.shap_values(X_sample_scaled)      # (500, 30)

# Model 2: XGB (Explanation -> values)
sv2 = explainer2(X_sample_scaled)                           # Explanation
shap_values2 = sv2.values                                   # (500, 30)

# Model 3: RF (keep full Explanation, we will slice later)
sv3 = explainer3(X_sample_scaled)                           # Explanation (500, 2, 30) internally

print("shap_values1:", np.array(shap_values1).shape)
print("shap_values2:", np.array(shap_values2).shape)
print("X_sample_df:", X_sample_df.shape)


fraud_indices = np.where(y_test.values == 1)[0]
legit_indices = np.where(y_test.values == 0)[0]
fraud_idx     = fraud_indices[0]
legit_idx     = legit_indices[0]


SHAP ANALYSIS
shap_values1: (500, 30)
shap_values2: (500, 30)
X_sample_df: (500, 30)


In [ ]:
print("shap_values1:", np.array(shap_values1).shape)
print("shap_values2:", np.array(shap_values2).shape)
print("shap_fraud3:", np.array(shap_fraud3).shape)
print("X_sample_df:", X_sample_df.shape)


shap_values1: (500, 30)
shap_values2: (500, 30)
shap_fraud3: (500, 2)
X_sample_df: (500, 30)


In [56]:
# Model 1: Logistic Regression
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values1, X_sample_df, feature_names=feature_names,
                  plot_type="bar", show=False)
plt.title("SHAP Global Feature Importance — Model 1 (Logistic Regression)")
save_fig("shap1_bar.png")

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values1, X_sample_df, feature_names=feature_names,
                  show=False)
plt.title("SHAP Beeswarm — Model 1 (Logistic Regression)")
save_fig("shap1_beeswarm.png")

# Model 2: XGBoost
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values2, X_sample_df, feature_names=feature_names,
                  plot_type="bar", show=False)
plt.title("SHAP Global Feature Importance — Model 2 (XGBoost)")
save_fig("shap2_bar.png")

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values2, X_sample_df, feature_names=feature_names,
                  show=False)
plt.title("SHAP Beeswarm — Model 2 (XGBoost)")
save_fig("shap2_beeswarm.png")

# Model 3: Random Forest — fraud class (class index 1)
sv3_fraud = sv3[:, 1]      # this is an Explanation with values shape (500, 30)

plt.figure(figsize=(10, 8))
shap.summary_plot(sv3_fraud, plot_type="bar", show=False)   # pass Explanation ONLY
plt.title("SHAP Global Feature Importance — Model 3 (Random Forest)")
save_fig("shap3_bar.png")

plt.figure(figsize=(10, 8))
shap.summary_plot(sv3_fraud, show=False)
plt.title("SHAP Beeswarm — Model 3 (Random Forest)")
save_fig("shap3_beeswarm.png")


  Saved: shap1_bar.png
  Saved: shap1_beeswarm.png
  Saved: shap2_bar.png
  Saved: shap2_beeswarm.png
  Saved: shap3_bar.png
  Saved: shap3_beeswarm.png


In [ ]:
# SHAP WATERFALL: fraud instance 
# Model 1
sv1_single = explainer1.shap_values(X_test_scaled[fraud_idx:fraud_idx+1])
expl1 = shap.Explanation(values=sv1_single[0], base_values=explainer1.expected_value,
                          data=X_test.iloc[fraud_idx].values, feature_names=feature_names)
plt.figure(); shap.plots.waterfall(expl1, max_display=10, show=False)
plt.title("SHAP Waterfall — Fraud (Model 1 - Logistic Regression)"); save_fig("shap1_waterfall_fraud.png")

# Model 2
sv2_single = explainer2(X_test_scaled[fraud_idx:fraud_idx+1])
expl2 = shap.Explanation(values=sv2_single.values[0], base_values=sv2_single.base_values[0],
                          data=X_test.iloc[fraud_idx].values, feature_names=feature_names)
plt.figure(); shap.plots.waterfall(expl2, max_display=10, show=False)
plt.title("SHAP Waterfall — Fraud (Model 2 - XGBoost)"); save_fig("shap2_waterfall_fraud.png")

# Model 3 — Random Forest, NEW API
sv3_single = explainer3(X_test_scaled[fraud_idx:fraud_idx+1])   # Explanation, shape (1, 2, 30)
values3_single = sv3_single.values[0, 1, :]                     # fraud class = index 1
base3_single   = sv3_single.base_values[0, 1]

expl3 = shap.Explanation(
    values=values3_single,
    base_values=base3_single,
    data=X_test.iloc[fraud_idx].values,
    feature_names=feature_names
)
plt.figure()
shap.plots.waterfall(expl3, max_display=10, show=False)
plt.title("SHAP Waterfall — Fraud (Model 3 - Random Forest)")
save_fig("shap3_waterfall_fraud.png")

# SHAP WATERFALL: legitimate instance (Random Forest)
sv3_legit = explainer3(X_test_scaled[legit_idx:legit_idx+1])
values3_legit = sv3_legit.values[0, 1, :]
base3_legit   = sv3_legit.base_values[0, 1]

expl3_legit = shap.Explanation(
    values=values3_legit,
    base_values=base3_legit,
    data=X_test.iloc[legit_idx].values,
    feature_names=feature_names
)
plt.figure()
shap.plots.waterfall(expl3_legit, max_display=10, show=False)
plt.title("SHAP Waterfall — Legitimate (Model 3 - Random Forest)")
save_fig("shap3_waterfall_legit.png")

  Saved: shap1_waterfall_fraud.png
  Saved: shap2_waterfall_fraud.png
  Saved: shap3_waterfall_fraud.png
  Saved: shap3_waterfall_legit.png


In [ ]:
# LIME EXPLANATIONS
print("\n" + "="*65)
print("LIME ANALYSIS")
print("="*65)

lime_explainer = lime_tabular.LimeTabularExplainer(
    training_data=X_train_res,
    feature_names=feature_names,
    class_names=['Legitimate', 'Fraud'],
    mode='classification',
    random_state=42
)

fraud_instance = X_test_scaled[fraud_idx]
legit_instance = X_test_scaled[legit_idx]

for model, mname, fname_prefix in [
    (model1, 'Model 1 (Logistic Regression)', 'lime1'),
    (model2, 'Model 2 (XGBoost)',             'lime2'),
    (model3, 'Model 3 (Random Forest)',       'lime3'),
]:
    # Fraud instance
    exp_f = lime_explainer.explain_instance(fraud_instance, model.predict_proba, num_features=10)
    fig   = exp_f.as_pyplot_figure()
    fig.suptitle(f"LIME — Fraud Instance ({mname})", fontsize=11, y=1.02)
    fig.tight_layout()
    fig.savefig(f"{fname_prefix}_fraud.png", dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"  Saved: {fname_prefix}_fraud.png")

    # Legitimate instance
    exp_l = lime_explainer.explain_instance(legit_instance, model.predict_proba, num_features=10)
    fig   = exp_l.as_pyplot_figure()
    fig.suptitle(f"LIME — Legitimate Instance ({mname})", fontsize=11, y=1.02)
    fig.tight_layout()
    fig.savefig(f"{fname_prefix}_legit.png", dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"  Saved: {fname_prefix}_legit.png")



LIME ANALYSIS
  Saved: lime1_fraud.png
  Saved: lime1_legit.png
  Saved: lime2_fraud.png
  Saved: lime2_legit.png
  Saved: lime3_fraud.png
  Saved: lime3_legit.png


In [ ]:
# EXPLANATION STABILITY ANALYSIS  (RQ1)
print("\n" + "="*65)
print("EXPLANATION STABILITY ANALYSIS (RQ1)")
print("="*65)

N_RUNS     = 10
N_FEATURES = 10

# LIME stability
print("\n  LIME stability — 10 independent runs on the same fraud instance")
print("  (LIME is stochastic by design; stability = low rank variance across runs)\n")

all_lime_rankings = []  # shape: (N_RUNS, N_FEATURES)

for run in range(N_RUNS):
    # Different random_state each run to simulate real stochasticity
    lime_exp_r = lime_tabular.LimeTabularExplainer(
        training_data=X_train_res,
        feature_names=feature_names,
        class_names=['Legitimate', 'Fraud'],
        mode='classification',
        random_state=run * 7   # vary seed deliberately
    )
    exp_r = lime_exp_r.explain_instance(
        fraud_instance, model3.predict_proba, num_features=N_FEATURES
    )
    # Extract feature names in rank order (LIME returns sorted by |weight|)
    ranked_features = [feat for feat, _ in exp_r.as_list()]
    # Clean up feature names (LIME adds inequalities like "V14 <= -5.3")
    ranked_features = [f.split(' ')[0].replace('V', 'V') for f in ranked_features]
    all_lime_rankings.append(ranked_features)


EXPLANATION STABILITY ANALYSIS (RQ1)

  LIME stability — 10 independent runs on the same fraud instance
  (LIME is stochastic by design; stability = low rank variance across runs)



In [61]:
# Build rank matrix: for each feature, what rank did it appear in each run?
unique_features = list(set(f for run in all_lime_rankings for f in run))
rank_matrix = {feat: [] for feat in unique_features}
for run_ranks in all_lime_rankings:
    for rank, feat in enumerate(run_ranks, 1):
        rank_matrix[feat].append(rank)
    # Features not in top-10 for this run get rank 11
    missing = [f for f in unique_features if f not in run_ranks]
    for feat in missing:
        rank_matrix[feat].append(N_FEATURES + 1)

stability_rows = []
for feat in unique_features:
    ranks = rank_matrix[feat]
    stability_rows.append({
        'Feature':    feat,
        'Mean rank':  round(np.mean(ranks), 2),
        'Std rank':   round(np.std(ranks), 2),
        'Appearances': sum(1 for r in ranks if r <= N_FEATURES),
    })

stability_df = pd.DataFrame(stability_rows).sort_values('Mean rank')
print(stability_df.to_string(index=False))
stability_df.to_csv('lime_stability.csv', index=False)
print("\n  Saved: lime_stability.csv")


Feature  Mean rank  Std rank  Appearances
    V14        1.0      0.00           10
    V12        2.4      0.49           10
    V17        2.6      0.49           10
    V11        4.0      0.00           10
    V16        5.2      0.40           10
    V10        6.4      0.80           10
     V3        7.0      0.77           10
   1.01        7.4      0.92           10
    V18        9.3      0.64            9
     V9       10.3      0.64            6
     V6       10.7      0.46            3
     V7       10.8      0.60            1
    V20       10.9      0.30            1

  Saved: lime_stability.csv


In [62]:
# Average rank standard deviation across top features (headline stability metric)
top_features_std = stability_df[stability_df['Appearances'] >= N_RUNS * 0.7]['Std rank']
mean_std = top_features_std.mean()
print(f"\n  Mean rank std (features appearing in ≥70% of runs): {mean_std:.3f}")
print(f"  Interpretation: lower = more stable LIME explanations")



  Mean rank std (features appearing in ≥70% of runs): 0.501
  Interpretation: lower = more stable LIME explanations


In [63]:
# Visualise LIME stability
fig, ax = plt.subplots(figsize=(9, 5))
top10 = stability_df.head(10)
ax.barh(top10['Feature'][::-1], top10['Std rank'][::-1], color='steelblue', alpha=0.8)
ax.set_xlabel('Rank standard deviation (10 runs)')
ax.set_title('LIME Explanation Stability — Random Forest\n'
             'Feature rank variance across 10 independent runs on same fraud instance')
ax.axvline(x=1.0, color='crimson', linestyle='--', alpha=0.7, label='Std = 1 (reference)')
ax.legend()
save_fig('lime_stability_plot.png')

  Saved: lime_stability_plot.png


In [ ]:
# SHAP stability (perturbation test)
print("\n  SHAP stability — perturbation test on fraud instance")
print("  Adds Gaussian noise (σ=0.05) 10 times, measures rank consistency\n")

N_PERTURB   = 10
NOISE_SIGMA = 0.05

# Base SHAP values for RF, fraud class only (class index 1)
sv3_base = explainer3(X_test_scaled[fraud_idx:fraud_idx+1])   # Explanation: (1, 2, 30)
base_vals = sv3_base.values[0, 1, :]                          # (30,)
base_ranking = np.argsort(np.abs(base_vals))[::-1][:N_FEATURES]

perturb_rankings = []

for _ in range(N_PERTURB):
    noise = np.random.normal(0, NOISE_SIGMA, X_test_scaled[fraud_idx].shape)
    perturbed = X_test_scaled[fraud_idx:fraud_idx+1] + noise

    sv3_pert = explainer3(perturbed)                          # Explanation: (1, 2, 30)
    vals_pert = sv3_pert.values[0, 1, :]                      # (30,)
    ranking_p = np.argsort(np.abs(vals_pert))[::-1][:N_FEATURES]
    perturb_rankings.append(ranking_p)


  SHAP stability — perturbation test on fraud instance
  Adds Gaussian noise (σ=0.05) 10 times, measures rank consistency



In [66]:
# Rank overlap with base (Jaccard-like: |intersection| / N_FEATURES)
overlaps = [len(set(base_ranking) & set(r)) / N_FEATURES for r in perturb_rankings]
mean_overlap = np.mean(overlaps)
std_overlap  = np.std(overlaps)

print(f"  SHAP top-{N_FEATURES} feature overlap with base (mean ± std): "
      f"{mean_overlap:.3f} ± {std_overlap:.3f}")
print(f"  Interpretation: 1.0 = identical top features across all perturbations")
print(f"  Result: SHAP is {'highly stable' if mean_overlap >= 0.85 else 'moderately stable'} "
      f"under small input perturbations")

# Save SHAP stability results
shap_stability = {
    'Noise sigma': NOISE_SIGMA,
    'N perturbations': N_PERTURB,
    'Mean overlap': round(mean_overlap, 4),
    'Std overlap':  round(std_overlap, 4),
    'Per-run overlaps': [round(o, 4) for o in overlaps]
}
pd.DataFrame([{k: v for k, v in shap_stability.items() if k != 'Per-run overlaps'}]).to_csv(
    'shap_stability_summary.csv', index=False)
print("  Saved: shap_stability_summary.csv")

  SHAP top-10 feature overlap with base (mean ± std): 0.200 ± 0.000
  Interpretation: 1.0 = identical top features across all perturbations
  Result: SHAP is moderately stable under small input perturbations
  Saved: shap_stability_summary.csv


In [ ]:
# LATENCY BENCHMARKING  (RQ2)
# Measures time for SHAP and LIME explanation generation per instance
print("\n" + "="*65)
print("LATENCY BENCHMARKING (RQ2)")
print("="*65)

BENCH_REPS = 5
single_instance = X_test_scaled[fraud_idx:fraud_idx+1]

latency_rows = []

print("\n  Benchmarking SHAP latency (per-instance)...")



LATENCY BENCHMARKING (RQ2)

  Benchmarking SHAP latency (per-instance)...


In [ ]:
# SHAP Model 1
times = []
for _ in range(BENCH_REPS):
    t0 = time.perf_counter()
    _ = explainer1.shap_values(single_instance)
    times.append((time.perf_counter() - t0) * 1000)
shap1_ms = round(np.mean(times), 2)
print(f"  SHAP Model 1 (LinearExplainer): {shap1_ms} ms")
latency_rows.append({'Method': 'SHAP', 'Model': 'Logistic Regression',
                     'Explainer': 'LinearExplainer', 'Latency_ms': shap1_ms})

# SHAP Model 2 
times = []
for _ in range(BENCH_REPS):
    t0 = time.perf_counter()
    _ = explainer2.shap_values(single_instance)
    times.append((time.perf_counter() - t0) * 1000)
shap2_ms = round(np.mean(times), 2)
print(f"  SHAP Model 2 (TreeExplainer/XGB): {shap2_ms} ms")
latency_rows.append({'Method': 'SHAP', 'Model': 'XGBoost',
                     'Explainer': 'TreeExplainer', 'Latency_ms': shap2_ms})

# SHAP Model 3 
times = []
for _ in range(BENCH_REPS):
    t0 = time.perf_counter()
    _ = explainer3.shap_values(single_instance)
    times.append((time.perf_counter() - t0) * 1000)
shap3_ms = round(np.mean(times), 2)
print(f"  SHAP Model 3 (TreeExplainer/RF): {shap3_ms} ms")
latency_rows.append({'Method': 'SHAP', 'Model': 'Random Forest',
                     'Explainer': 'TreeExplainer', 'Latency_ms': shap3_ms})

print("\n  Benchmarking LIME latency (per-instance)...")


  SHAP Model 1 (LinearExplainer): 0.14 ms
  SHAP Model 2 (TreeExplainer/XGB): 4.55 ms
  SHAP Model 3 (TreeExplainer/RF): 46.74 ms

  Benchmarking LIME latency (per-instance)...


In [69]:
# LIME is expensive per instance (perturbs + fits surrogate)
for model, mname in [(model1, 'Logistic Regression'), (model2, 'XGBoost'), (model3, 'Random Forest')]:
    times = []
    for _ in range(BENCH_REPS):
        t0 = time.perf_counter()
        lime_explainer.explain_instance(fraud_instance, model.predict_proba, num_features=10)
        times.append((time.perf_counter() - t0) * 1000)
    lime_ms = round(np.mean(times), 2)
    print(f"  LIME {mname}: {lime_ms} ms")
    latency_rows.append({'Method': 'LIME', 'Model': mname,
                         'Explainer': 'LimeTabularExplainer', 'Latency_ms': lime_ms})

latency_df = pd.DataFrame(latency_rows)
print("\n  Latency summary:")
print(latency_df.to_string(index=False))
latency_df.to_csv('latency_benchmark.csv', index=False)
print("  Saved: latency_benchmark.csv")

  LIME Logistic Regression: 110.95 ms
  LIME XGBoost: 145.79 ms
  LIME Random Forest: 223.99 ms

  Latency summary:
Method               Model            Explainer  Latency_ms
  SHAP Logistic Regression      LinearExplainer        0.14
  SHAP             XGBoost        TreeExplainer        4.55
  SHAP       Random Forest        TreeExplainer       46.74
  LIME Logistic Regression LimeTabularExplainer      110.95
  LIME             XGBoost LimeTabularExplainer      145.79
  LIME       Random Forest LimeTabularExplainer      223.99
  Saved: latency_benchmark.csv


In [70]:
# Visualise latency
fig, ax = plt.subplots(figsize=(9, 5))
colors = {'SHAP': 'steelblue', 'LIME': 'coral'}
x_pos = np.arange(len(latency_df))
bars = ax.bar(x_pos, latency_df['Latency_ms'],
              color=[colors[m] for m in latency_df['Method']], alpha=0.85)

ax.set_xticks(x_pos)
ax.set_xticklabels(
    [f"{row['Method']}\n{row['Model'].replace(' (Tuned)','')}".replace('Logistic Regression','LR')
     for _, row in latency_df.iterrows()],
    fontsize=9
)
ax.set_ylabel('Latency (ms per instance)')
ax.set_title('Explanation Latency — SHAP vs LIME per Model\n'
             f'(Average over {BENCH_REPS} runs, single fraud instance)')

Text(0.5, 1.0, 'Explanation Latency — SHAP vs LIME per Model\n(Average over 5 runs, single fraud instance)')

In [71]:
# Legend
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='steelblue', label='SHAP'),
                   Patch(color='coral', label='LIME')], fontsize=10)

# Annotate bars
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{bar.get_height():.1f}ms", ha='center', va='bottom', fontsize=8)

ax.grid(axis='y', alpha=0.3)
save_fig('latency_benchmark.png')

  Saved: latency_benchmark.png


In [ ]:
# COMBINED SUMMARY — all metrics for the dissertation results table
print("\n" + "="*65)
print("FINAL SUMMARY FOR DISSERTATION")
print("="*65)

# Recompute at optimal thresholds
summary_rows = []
for model, name, proba, thresh_res in [
    (model1, 'LR (baseline)',       y_proba1, thresh_results1),
    (model2, 'XGBoost (tuned)',     y_proba2, thresh_results2),
    (model3, 'Random Forest (tuned)',y_proba3, thresh_results3),
]:
    bt = thresh_res['best_thresh']
    y_pred_opt = (proba >= bt).astype(int)
    m = evaluate_model(name, y_test, y_pred_opt, proba)
    shap_lat = latency_df[(latency_df['Method']=='SHAP') &
                           (latency_df['Model'].str.contains(name.split()[0]))]['Latency_ms'].values
    lime_lat  = latency_df[(latency_df['Method']=='LIME') &
                           (latency_df['Model'].str.contains(name.split()[0]))]['Latency_ms'].values
    m['Opt_Threshold'] = round(bt, 3)
    m['SHAP_ms']       = shap_lat[0] if len(shap_lat) > 0 else None
    m['LIME_ms']       = lime_lat[0] if len(lime_lat) > 0 else None
    summary_rows.append(m)

summary_df = pd.DataFrame(summary_rows)
print(summary_df[['Model','ROC-AUC','PR-AUC','F1-Fraud','Precision',
                   'Recall','FP','FN','Opt_Threshold','SHAP_ms','LIME_ms']].to_string(index=False))
summary_df.to_csv('dissertation_results_summary.csv', index=False)
print("\n  Saved: dissertation_results_summary.csv")

print("\n" + "="*65)
print("ALL DONE — files generated:")
print("  Models:     ConfusionMAT1/2/3.png, ConfusionMAT3_optimal.png")
print("  Curves:     pr_curves.png, roc_curves.png")
print("  Threshold:  threshold_analysis_rf.png")
print("  SHAP:       shap1/2/3_bar.png, _beeswarm.png, _waterfall_fraud.png")
print("              shap3_waterfall_legit.png")
print("  LIME:       lime1/2/3_fraud.png, lime1/2/3_legit.png")
print("  Stability:  lime_stability.csv, lime_stability_plot.png")
print("              shap_stability_summary.csv")
print("  Latency:    latency_benchmark.csv, latency_benchmark.png")
print("  Tables:     model_comparison_default.csv, model_comparison_optimal.csv")
print("              dissertation_results_summary.csv")
print("="*65)


FINAL SUMMARY FOR DISSERTATION
                Model  ROC-AUC  PR-AUC  F1-Fraud  Precision  Recall  FP  FN  Opt_Threshold  SHAP_ms  LIME_ms
        LR (baseline)   0.9893  0.5848    0.7711     0.7442   0.800  11   8           1.00      NaN      NaN
      XGBoost (tuned)   0.9919  0.9613    0.9487     0.9737   0.925   1   3           0.84     4.55   145.79
Random Forest (tuned)   0.9847  0.9628    0.9367     0.9487   0.925   2   3           0.52    46.74   223.99

  Saved: dissertation_results_summary.csv

ALL DONE — files generated:
  Models:     ConfusionMAT1/2/3.png, ConfusionMAT3_optimal.png
  Curves:     pr_curves.png, roc_curves.png
  Threshold:  threshold_analysis_rf.png
  SHAP:       shap1/2/3_bar.png, _beeswarm.png, _waterfall_fraud.png
              shap3_waterfall_legit.png
  LIME:       lime1/2/3_fraud.png, lime1/2/3_legit.png
  Stability:  lime_stability.csv, lime_stability_plot.png
              shap_stability_summary.csv
  Latency:    latency_benchmark.csv, latency_bench